# Surrogate Selection — L5B15

Picks the surrogate-model architecture used for the rest of the thesis. The comparison is run on **L5B15** (primary luggage variant, largest sample) and benchmarks five families: Linear Regression, GaussianNB (architecture-matched to Pega ADM), RandomForest, LightGBM, and CatBoost — all on the same train/test split and the same Pega ADM target.

Outcome of §7.6: **CatBoost** is the selected surrogate. The chosen architecture is then refit per offer variant in `05_surrogate_fit.ipynb`, which is where downstream notebooks (06–08) read their model artifacts from.

**Run `02_data_ingestion.ipynb` first.**

In [1]:
# ── Config ────────────────────────────────────────────────────────────────
# Selection is L5B15-only. Per-variant fitting lives in 05_surrogate_fit.ipynb.
from pathlib import Path

VARIANT       = "L5B15"
PROCESSED_DIR = Path("../data/processed")
PROCESSED_FILE = PROCESSED_DIR / "luggage_email_outbound.parquet"
ARTIFACT_DIR  = Path("../data/artifacts") / VARIANT
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Variant : {VARIANT}")
print(f"Input   : {PROCESSED_FILE}")
print(f"Artifacts → {ARTIFACT_DIR}  (only the comparison CSV is written here from this notebook)")

Variant : L5B15
Input   : ../data/processed/luggage_email_outbound.parquet
Artifacts → ../data/artifacts/L5B15  (only the comparison CSV is written here from this notebook)


In [2]:
%load_ext autoreload
%autoreload 2

import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.insert(0, "../src")
from my_project.features import VARIANT_FEATURES
from my_project.metrics import fidelity_suite
from my_project.surrogate import (
    NaiveBayesBaseline,
    build_feature_matrix,
    cv_select_depth,
    evaluate_surrogate,
    train_catboost,
)

print("Imports OK")

Imports OK


In [3]:
df = pd.read_parquet(PROCESSED_FILE)
# Filter to primary variant AND the production-decision technique.
# Every scoring event in the export is evaluated by two parallel techniques —
# the production model (label degenerate to "0.0", confirmed with Transavia
# data team as the model that actually drives offer decisions) and a
# NaiveBayes audit/shadow model. We restrict downstream analysis to the
# production model. See 03_eda §6.10 for rationale + breakdown.
df = df[(df["pyName"] == VARIANT) & (df["modelTechnique"] == "0.0")].reset_index(drop=True)
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
print(f"Model versions: {df['modelVersion'].nunique()}")
print(df["modelVersion"].value_counts().head(10).to_string())

Rows: 31,709  |  Columns: 442
Model versions: 575
modelVersion
ab6b5447-9b9d-5719-b150-373ab497023b    697
488e6625-2e97-5eba-849a-c8838455235d    665
eb8ffcb9-18d6-589a-9aec-9fb764a00190    660
9b0531bc-a651-52be-8759-033ba3cab48f    660
010bb271-5694-5318-9918-794f4264cd05    658
d28130d1-8da2-5700-86b7-cb87cc69b60f    635
8f684d3d-0e3e-55a7-b4f2-85a9ea386989    600
08060c8b-f119-5d39-bd3c-7b60cb327735    598
c898d9fc-601c-5044-b056-3adae2ad9901    585
e9c04173-2c4c-525c-88d6-2e8355912a51    570


## 7. Surrogate Model

The surrogate approximates the black-box Pega ADM function $f$ using only the logged tuples $(X_i, \hat{p}_i)$. It serves as the prediction oracle for SHAP and LIME in notebook 05. Fidelity is measured on a held-out test set.

In [4]:
### 7.1  Feature matrix
cfg = VARIANT_FEATURES[VARIANT]
X, y, cat_cols, num_cols = build_feature_matrix(df, list(cfg.features), cfg.numeric)
print(f"Features: {X.shape[1]}  ({len(num_cols)} numeric, {len(cat_cols)} categorical)")
print(f"Target   : {y.describe().to_dict()}")
X.head(3)

Features: 23  (9 numeric, 14 categorical)
Target   : {'count': 31709.0, 'mean': 0.007790698520194741, 'std': 0.011194852174158002, 'min': 0.0026520772043582552, '25%': 0.002904314556717299, '50%': 0.004767443136308276, '75%': 0.009001105071381637, 'max': 0.4049429657794677}


,CustBookedFlight.BookingData.BookingMonth,CustBookedFlight.BookingData.BookerGender,CustBookedFlight.BookingData.CultureCode,CustBookedFlight.BookingData.FlightInboundArrival,CustBookedFlight.Language,CustBookedFlight.Journey,CustBookedFlight.FlightNumberOperatorIATA,CustBookedFlight.SeatNumber,CustBookedFlight.IsStaffStandBy,CustBookedFlight.FlightData.AirlineCodeIATA,...,IH.Email.Outbound.Pending.pxLastGroupID,IH.Email.Outbound.Delivered.pxLastOutcomeTime.DaysSince,IH.Email.Outbound.Delivered.pxLastGroupID,IH.Email.Inbound.Pending.pxLastOutcomeTime.DaysSince,IH.Email.Inbound.Clicked.pxLastOutcomeTime.DaysSince,IH.Push.Outbound.Pending.pyHistoricalOutcomeCount,IH.Push.Outbound.Pending.pxLastOutcomeTime.DaysSince,IH.Push.Outbound.Pending.pxLastGroupID,IH.Event.Outbound.RealTimeEvent.pyHistoricalOutcomeCount,param::Param.BundleName
0,4.0,Male,fr-FR,ORY,French,1,TO,__MISSING__,__MISSING__,TO,...,__MISSING__,NaN,__MISSING__,NaN,NaN,NaN,NaN,__MISSING__,NaN,BookingConfirmation
1,4.0,Male,fr-FR,__MISSING__,French,1,TO,__MISSING__,__MISSING__,TO,...,__MISSING__,NaN,__MISSING__,NaN,NaN,NaN,NaN,__MISSING__,NaN,BookingConfirmation
2,4.0,Male,en-GB,__MISSING__,English,1,TO,__MISSING__,__MISSING__,TO,...,ThirdParty,0.005488,ThirdParty,NaN,NaN,3.0,2.113973,Seats,NaN,BookingConfirmation


In [5]:
### 7.2  Train / test split  (stratified 80/20 on propensity deciles, fixed seed)
# Stratifying on qcut(y) bins keeps both halves' propensity distributions matched
# so the held-out fidelity metrics (especially KS + the rank metrics) score the
# surrogate on the full propensity range. Temporal splits for stability analysis
# are defined in notebook 07.
bins = pd.qcut(y, q=10, labels=False, duplicates="drop")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=bins,
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

Train: 25,367  |  Test: 6,342


In [6]:
### 7.3  CatBoost — depth grid search (5-fold CV on Spearman ρ)
# The NB generating process is strictly additive (no feature interactions),
# so lower depths should generalise better. CV ranges over depths 1..15 to
# clearly avoid edge artefacts (CatBoost guidance is depths ≤ 10 are usually
# sufficient; we extend past that to confirm the curve plateaus). Selection
# uses Breiman's 1-SD rule (smallest depth within one fold-to-fold std of
# the max) to prefer parsimony when gains from deeper trees are within
# noise. Same routine is reused per variant in 05_surrogate_fit.
best_depth, cv_df = cv_select_depth(X_train, y_train, cat_cols)
max_depth = int(cv_df["mean_rho"].idxmax())   # CV-optimal (may differ from 1-SD pick)

display(
    cv_df.style.format("{:.4f}")
    .highlight_max(subset=["mean_rho"], color="#fff3cd")          # yellow = CV-max
    .apply(lambda s: ["background-color: #d4edda" if i == best_depth else "" for i in s.index],
           subset=["mean_rho"])                                    # green  = 1-SD pick
)
print(f"CV-max depth      : {max_depth}  (mean ρ = {cv_df.loc[max_depth, 'mean_rho']:.4f})")
print(f"Selected (1-SD)   : {best_depth}  (mean ρ = {cv_df.loc[best_depth, 'mean_rho']:.4f})")

cb_model    = train_catboost(X_train, y_train, cat_cols, depth=best_depth)
cb_fidelity = evaluate_surrogate(cb_model, X_test, y_test, f"CatBoost (depth={best_depth})")

,mean_rho,std_rho
depth,,
1,0.8299,0.0058
2,0.8920,0.0012
3,0.9029,0.0018
4,0.9068,0.0021
5,0.9108,0.0026
6,0.9126,0.0041
7,0.9172,0.0034
8,0.9200,0.0019
9,0.9223,0.0025


CV-max depth      : 12  (mean ρ = 0.9255)
Selected (1-SD)   : 10  (mean ρ = 0.9238)


### 7.3b  Depth selection sensitivity analysis (thesis appendix)

The 1-SD parsimony rule's pick can shift by ±1 depth as the CV range upper bound is extended. This cell reruns the CV at the narrower range 1–12 and compares to the §7.3 range 1–15 results to quantify that sensitivity. It produces the table the thesis chapter cites as evidence that — while the integer pick is mildly range-dependent — the corresponding CV mean ρ for any candidate depth differs from the next by less than one fold-to-fold std, so the resulting surrogate quality is range-invariant within CV noise.

Reuses `cv_df` and `best_depth` from §7.3 for the range-1-15 column; only re-runs CV for the range-1-12 column. Saves the full CV results (`depth_cv_results.csv`) and the joined sensitivity table (`depth_sensitivity_table.csv`) under `data/artifacts/L5B15/`.

In [7]:
# Reuse the range-1-15 CV results from §7.3 (no need to recompute).
cv_df_wide = cv_df.copy()
best_wide  = best_depth

# Re-run CV with the narrower range 1-12 for comparison.
best_narrow, cv_df_narrow = cv_select_depth(
    X_train, y_train, cat_cols, depths=range(1, 13),
)

# 1-SD thresholds at each range's CV-max.
max_wide   = int(cv_df_wide["mean_rho"].idxmax())
max_narrow = int(cv_df_narrow["mean_rho"].idxmax())
threshold_wide   = cv_df_wide.loc[max_wide, "mean_rho"]     - cv_df_wide.loc[max_wide, "std_rho"]
threshold_narrow = cv_df_narrow.loc[max_narrow, "mean_rho"] - cv_df_narrow.loc[max_narrow, "std_rho"]

print(f"Range 1-12  →  CV-max depth {max_narrow}  (ρ={cv_df_narrow.loc[max_narrow,'mean_rho']:.4f}, "
      f"std={cv_df_narrow.loc[max_narrow,'std_rho']:.4f})  |  1-SD threshold = {threshold_narrow:.4f}  |  "
      f"Selected depth = {best_narrow}")
print(f"Range 1-15  →  CV-max depth {max_wide}  (ρ={cv_df_wide.loc[max_wide,'mean_rho']:.4f}, "
      f"std={cv_df_wide.loc[max_wide,'std_rho']:.4f})  |  1-SD threshold = {threshold_wide:.4f}  |  "
      f"Selected depth = {best_wide}")

# Build the joined sensitivity table (one row per depth, two columns of CV results).
sensitivity_df = pd.DataFrame({
    "Depth":           cv_df_wide.index.astype(int),
    "Mean ρ (1-15)":   cv_df_wide["mean_rho"].values,
    "Std ρ (1-15)":    cv_df_wide["std_rho"].values,
    "Mean ρ (1-12)":   cv_df_narrow["mean_rho"].reindex(cv_df_wide.index).values,
    "Selected (1-12)": ["yes" if d == best_narrow else "no" for d in cv_df_wide.index],
    "Selected (1-15)": ["yes" if d == best_wide   else "no" for d in cv_df_wide.index],
})

display(
    sensitivity_df.style
    .format({"Mean ρ (1-15)": "{:.4f}", "Std ρ (1-15)": "{:.4f}",
             "Mean ρ (1-12)": lambda x: "—" if pd.isna(x) else f"{x:.4f}"})
    .hide(axis="index")
    .set_caption(f"1-SD threshold under range 1-12 = {threshold_narrow:.4f}; "
                 f"under range 1-15 = {threshold_wide:.4f}")
)

# Persist for the thesis.
cv_df_wide.to_csv(ARTIFACT_DIR / "depth_cv_results.csv")
sensitivity_df.to_csv(ARTIFACT_DIR / "depth_sensitivity_table.csv", index=False)
print(f"\nSaved → {ARTIFACT_DIR / 'depth_cv_results.csv'}")
print(f"Saved → {ARTIFACT_DIR / 'depth_sensitivity_table.csv'}")

Range 1-12  →  CV-max depth 12  (ρ=0.9255, std=0.0022)  |  1-SD threshold = 0.9233  |  Selected depth = 10
Range 1-15  →  CV-max depth 12  (ρ=0.9255, std=0.0022)  |  1-SD threshold = 0.9233  |  Selected depth = 10


Depth,Mean ρ (1-15),Std ρ (1-15),Mean ρ (1-12),Selected (1-12),Selected (1-15)
1,0.8299,0.0058,0.8299,no,no
2,0.8920,0.0012,0.8920,no,no
3,0.9029,0.0018,0.9029,no,no
4,0.9068,0.0021,0.9068,no,no
5,0.9108,0.0026,0.9108,no,no
6,0.9126,0.0041,0.9126,no,no
7,0.9172,0.0034,0.9172,no,no
8,0.9200,0.0019,0.9200,no,no
9,0.9223,0.0025,0.9223,no,no
10,0.9238,0.0027,0.9238,yes,yes



Saved → ../data/artifacts/L5B15/depth_cv_results.csv
Saved → ../data/artifacts/L5B15/depth_sensitivity_table.csv


In [8]:
# LaTeX export for the thesis. pandas 3.x removed `booktabs=` from
# DataFrame.to_latex; we use Styler.to_latex(hrules=True), which emits
# \toprule/\midrule/\bottomrule. caption/label set on the Styler.
latex_table = (
    sensitivity_df.style
    .format({"Mean ρ (1-15)": "{:.4f}", "Std ρ (1-15)": "{:.4f}",
             "Mean ρ (1-12)": lambda x: "--" if pd.isna(x) else f"{x:.4f}"})
    .hide(axis="index")
    .set_caption(
        "Five-fold cross-validation Spearman~$\\rho$ for CatBoost depth "
        "selection on L5B15. The 1-SD band is defined as the CV maximum "
        "mean~$\\rho$ minus one fold-to-fold standard deviation. Selected "
        "depth under each range bound is indicated."
    )
    .to_latex(hrules=True, label="tab:depth_sensitivity")
)
print(latex_table)

\begin{table}
\caption{Five-fold cross-validation Spearman~$\rho$ for CatBoost depth selection on L5B15. The 1-SD band is defined as the CV maximum mean~$\rho$ minus one fold-to-fold standard deviation. Selected depth under each range bound is indicated.}
\label{tab:depth_sensitivity}
\begin{tabular}{rrrrll}
\toprule
Depth & Mean ρ (1-15) & Std ρ (1-15) & Mean ρ (1-12) & Selected (1-12) & Selected (1-15) \\
\midrule
1 & 0.8299 & 0.0058 & 0.8299 & no & no \\
2 & 0.8920 & 0.0012 & 0.8920 & no & no \\
3 & 0.9029 & 0.0018 & 0.9029 & no & no \\
4 & 0.9068 & 0.0021 & 0.9068 & no & no \\
5 & 0.9108 & 0.0026 & 0.9108 & no & no \\
6 & 0.9126 & 0.0041 & 0.9126 & no & no \\
7 & 0.9172 & 0.0034 & 0.9172 & no & no \\
8 & 0.9200 & 0.0019 & 0.9200 & no & no \\
9 & 0.9223 & 0.0025 & 0.9223 & no & no \\
10 & 0.9238 & 0.0027 & 0.9238 & yes & yes \\
11 & 0.9244 & 0.0016 & 0.9244 & no & no \\
12 & 0.9255 & 0.0022 & 0.9255 & no & no \\
13 & 0.9254 & 0.0028 & -- & no & no \\
14 & 0.9242 & 0.0022 & -- & no &

In [9]:
### 7.4  Naive Bayes robustness baseline
nb_model    = NaiveBayesBaseline(n_bins=10).fit(X_train, y_train, cat_cols, num_cols)
nb_fidelity = evaluate_surrogate(nb_model, X_test, y_test, "NaiveBayes")

In [10]:
### 7.5  Fidelity comparison
fidelity_df = (
    pd.DataFrame([cb_fidelity, nb_fidelity])
    .set_index("model")
    .rename(columns={
        "r2":           "R²",
        "rmse":         "RMSE",
        "spearman_rho": "Spearman ρ",
        "kendall_tau":  "Kendall τ",
        "ks_stat":      "KS",
    })
)
display(
    fidelity_df.style
    .format("{:.4f}")
    .highlight_max(subset=["R²", "Spearman ρ", "Kendall τ"], color="#d4edda")
    .highlight_min(subset=["RMSE", "KS"],                    color="#d4edda")
)

,R²,RMSE,Spearman ρ,Kendall τ,KS
model,,,,,
CatBoost (depth=10),0.9266,0.0026,0.9327,0.7985,0.1539
NaiveBayes,0.3678,0.0077,0.8618,0.6579,0.2679


### 7.6  Surrogate method comparison

Five surrogate families on the same train/test split and the same Pega ADM target:

- **Linear Regression** — lower-bound baseline (no interactions in the encoded space).
- **GaussianNB** — architecture-matched to Pega ADM's internal Naive Bayes. Target is binned into deciles and the classifier's probabilities are projected back to continuous predictions via class-probability-weighted bin means.
- **Random Forest** — tree-based, no boosting; isolates whether boosting specifically or general tree flexibility drives fidelity.
- **LightGBM** — boosting with native categorical handling, comparable to CatBoost.
- **CatBoost** — primary surrogate, depth-tuned in §7.3.

Linear / GaussianNB / Random Forest / LightGBM use `OrdinalEncoder(unknown_value=-1)` on cat columns with mean-imputed numerics. LightGBM gets the cat indices via `categorical_feature`. CatBoost reuses `X_train`/`X_test` directly with native cat handling.

In [11]:
### 7.6.1  Shared encoded matrix for Linear / GaussianNB / RandomForest / LightGBM
# Cat columns first (so cat indices are 0..len(cat_cols)-1), numerics last with
# mean imputation. Unknown categories in the test set → -1 sentinel.
import lightgbm as lgb
from sklearn.ensemble    import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes  import GaussianNB
from sklearn.preprocessing import OrdinalEncoder

ord_enc      = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_tr_cat_enc = ord_enc.fit_transform(X_train[cat_cols].astype(str))
X_te_cat_enc = ord_enc.transform(X_test[cat_cols].astype(str))

num_means = X_train[num_cols].apply(pd.to_numeric, errors="coerce").mean()
X_tr_num  = (X_train[num_cols].apply(pd.to_numeric, errors="coerce")
                              .fillna(num_means).values)
X_te_num  = (X_test[num_cols].apply(pd.to_numeric, errors="coerce")
                             .fillna(num_means).values)

X_tr_enc    = np.hstack([X_tr_cat_enc, X_tr_num])
X_te_enc    = np.hstack([X_te_cat_enc, X_te_num])
cat_idx_enc = list(range(len(cat_cols)))  # cat cols occupy the first len(cat_cols) columns

print(f"Encoded matrix: train {X_tr_enc.shape}, test {X_te_enc.shape}")
print(f"Categorical indices (for LightGBM): {cat_idx_enc}")

Encoded matrix: train (25367, 23), test (6342, 23)
Categorical indices (for LightGBM): [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


In [12]:
### 7.6.2  Fit five surrogates on the same train/test split and collect fidelity rows.
# CatBoost reuses cb_model from §7.3 (already random_seed=42, verbose=False).
# All sklearn-style models use the encoded matrix; LightGBM gets cat indices natively.

results = []

# Linear Regression — lower-bound, no interactions in the encoded space
lr_pred = LinearRegression().fit(X_tr_enc, y_train).predict(X_te_enc)
results.append({"Model": "LinearRegression", **fidelity_suite(y_test.values, lr_pred)})

# GaussianNB — classifier; bin target into deciles, recover via weighted bin means
y_tr_bins = pd.qcut(y_train, q=10, labels=False, duplicates="drop").astype(int)
bin_means = y_train.groupby(y_tr_bins).mean().sort_index().values
gnb       = GaussianNB().fit(X_tr_enc, y_tr_bins)
gnb_pred  = gnb.predict_proba(X_te_enc) @ bin_means
results.append({"Model": "GaussianNB", **fidelity_suite(y_test.values, gnb_pred)})

# Random Forest — tree flexibility, no boosting
rf_pred = RandomForestRegressor(
    n_estimators=100, random_state=42, n_jobs=-1,
).fit(X_tr_enc, y_train).predict(X_te_enc)
results.append({"Model": "RandomForest", **fidelity_suite(y_test.values, rf_pred)})

# LightGBM — boosting with native categorical handling on the ordinal-coded matrix
lgb_pred = lgb.LGBMRegressor(random_state=42, verbose=-1).fit(
    X_tr_enc, y_train,
    categorical_feature=cat_idx_enc,
).predict(X_te_enc)
results.append({"Model": "LightGBM", **fidelity_suite(y_test.values, lgb_pred)})

# CatBoost — reuse depth-tuned model from §7.3
results.append({"Model": "CatBoost", **fidelity_suite(y_test.values, cb_model.predict(X_test))})

comparison_df = (
    pd.DataFrame(results)[["Model", "r2", "rmse", "spearman_rho", "kendall_tau", "ks_stat"]]
    .rename(columns={
        "r2":           "R²",
        "rmse":         "RMSE",
        "spearman_rho": "Spearman ρ",
        "kendall_tau":  "Kendall τ",
        "ks_stat":      "KS statistic",
    })
)

display(
    comparison_df.style
    .format({"R²": "{:.4f}", "RMSE": "{:.6f}",
             "Spearman ρ": "{:.4f}", "Kendall τ": "{:.4f}", "KS statistic": "{:.4f}"})
    .highlight_max(subset=["R²", "Spearman ρ", "Kendall τ"], color="#d4edda")
    .highlight_min(subset=["RMSE", "KS statistic"],          color="#d4edda")
    .hide(axis="index")
)

/Users/noah.bos/Documents/thesis-repo/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Model,R²,RMSE,Spearman ρ,Kendall τ,KS statistic
LinearRegression,0.4685,0.007037,0.7359,0.5339,0.2211
GaussianNB,0.3678,0.007675,0.8618,0.6579,0.2679
RandomForest,0.7614,0.004715,0.9341,0.7879,0.1386
LightGBM,0.8377,0.003889,0.9245,0.7793,0.0902
CatBoost,0.9266,0.002616,0.9327,0.7985,0.1539


In [13]:
### 7.6.3  LaTeX table + persisted CSV
# pandas 3.x removed `booktabs` from DataFrame.to_latex(); use the Styler path,
# which emits \toprule/\midrule/\bottomrule when hrules=True.
latex_table = (
    comparison_df.style
    .format({"R²": "{:.4f}", "RMSE": "{:.6f}",
             "Spearman ρ": "{:.4f}", "Kendall τ": "{:.4f}", "KS statistic": "{:.4f}"})
    .hide(axis="index")
    .to_latex(hrules=True)
)
print(latex_table)

comparison_path = ARTIFACT_DIR / "surrogate_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print(f"Saved → {comparison_path}")

\begin{tabular}{lrrrrr}
\toprule
Model & R² & RMSE & Spearman ρ & Kendall τ & KS statistic \\
\midrule
LinearRegression & 0.4685 & 0.007037 & 0.7359 & 0.5339 & 0.2211 \\
GaussianNB & 0.3678 & 0.007675 & 0.8618 & 0.6579 & 0.2679 \\
RandomForest & 0.7614 & 0.004715 & 0.9341 & 0.7879 & 0.1386 \\
LightGBM & 0.8377 & 0.003889 & 0.9245 & 0.7793 & 0.0902 \\
CatBoost & 0.9266 & 0.002616 & 0.9327 & 0.7985 & 0.1539 \\
\bottomrule
\end{tabular}

Saved → ../data/artifacts/L5B15/surrogate_comparison.csv
